# Hybrid Evaluation (ML + Rule-Based)

**Purpose:** compare the three detection systems — rule-based, ML, and hybrid — and
report *how often* the rule-based override actually changes an ML prediction.

## Reporting guarantee

Earlier iterations reported a "hybrid" system whose overrides changed **zero**
predictions. This notebook instead:

- reports **how many** test predictions the hybrid overrides actually change;
- breaks those changes into **helped / hurt / neutral** samples;
- shows per-allergen behaviour, with a focused **tree-nuts (rare class)** case study.

## Setup & Imports

In [11]:
import os
import json
import ast
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

from sklearn.metrics import f1_score, precision_score, recall_score

from utils.data_utils import get_data_directories
from utils.text_processing import get_allergen_list, rule_match
from utils.model_utils import load_model_and_tokenizer, load_hybrid_config, predict_ml, hybrid_predict
from utils.evaluation import print_classification_report, print_per_class_metrics

print("Imports complete.")

Imports complete.


## 1. Load Model, Test Split, Thresholds

In [12]:
dirs = get_data_directories()
split_dir = os.path.join(dirs["final"], "splits")

def load_split(name):
    d = pd.read_csv(os.path.join(split_dir, name))
    d["labels"] = d["labels"].apply(ast.literal_eval)
    if "text" not in d.columns:
        d["text"] = d["ingredients_cleaned"]
    return d

test_df = load_split("test.csv")
test_texts = test_df["text"].tolist()
test_labels = np.array(test_df["labels"].tolist())
allergen_list = get_allergen_list()

model, tokenizer, device = load_model_and_tokenizer(os.path.join(dirs["models"], "mobilebert_allergen_final"))
hybrid_config = load_hybrid_config(os.path.join(dirs["models"], "hybrid_config.json"))
ml_thresholds = np.array(hybrid_config["ml_thresholds"])
rule_conf = hybrid_config.get("rule_conf_threshold", 0.5)
max_length = hybrid_config.get("max_length", 209)
print("Model + hybrid config loaded.")

Model + hybrid config loaded.


## 2. Rule-Based Predictions (no ML)

In [13]:
def rule_predict(texts):
    preds = np.zeros((len(texts), len(allergen_list)), dtype=int)
    for i, t in enumerate(texts):
        for j, a in enumerate(allergen_list):
            preds[i, j] = 1 if rule_match(t, a) else 0
    return preds

rule_preds = rule_predict(test_texts)
print(f"Rule predictions: {rule_preds.sum(axis=1).mean():.2f} allergens/product on average")

Rule predictions: 1.53 allergens/product on average


## 3. ML Predictions

In [14]:
ml_preds, ml_probs = predict_ml(
    test_texts, model, tokenizer, device,
    thresholds=ml_thresholds, max_length=max_length, batch_size=8,
)
print(f"ML predictions: {ml_preds.sum(axis=1).mean():.2f} allergens/product on average")

ML predictions: 1.31 allergens/product on average


## 4. Hybrid Predictions

**Recall-first config:** `rule_priority` for **every** allergen — the hybrid is the *union*
of the rule engine and the ML model. A consumer is warned if either system catches an
allergen; a missed allergen (false negative) is the safety-critical error. The precision
cost of the union (more warnings, some spurious) is the explicit trade for recall-first
detection. (An earlier config used `hard_override` for common classes and `rule_priority`
only for tree_nuts / shellfish / fish.)

In [15]:
# Recall-first: rule_priority for ALL allergens (rule ∪ ML — max recall)
RECALL_FIRST_MODES = {a: "rule_priority" for a in allergen_list}
hybrid_preds, _ = hybrid_predict(
    test_texts, model, tokenizer, device,
    thresholds=ml_thresholds,
    rule_conf_threshold=rule_conf,
    mode="hard_override",
    per_allergen_modes=RECALL_FIRST_MODES,
    max_length=max_length, batch_size=8,
)
print(f"Hybrid predictions (rule_priority for all): {hybrid_preds.sum(axis=1).mean():.2f} allergens/product on average")

Hybrid predictions (rule_priority for all): 1.55 allergens/product on average


## 5. Three-Way Comparison

In [16]:
def summarize(name, preds):
    return {
        "name": name,
        "macro_f1": f1_score(test_labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(test_labels, preds, average="micro", zero_division=0),
        "samples_f1": f1_score(test_labels, preds, average="samples", zero_division=0),
        "avg_preds": preds.sum(axis=1).mean(),
    }

rows = [summarize("Rule", rule_preds), summarize("ML", ml_preds), summarize("Hybrid", hybrid_preds)]
print(f"{'System':<10} {'Macro F1':>9} {'Micro F1':>9} {'Samples F1':>11} {'Avg preds':>9}")
print("-" * 50)
for r in rows:
    print(f"{r['name']:<10} {r['macro_f1']:>9.4f} {r['micro_f1']:>9.4f} {r['samples_f1']:>11.4f} {r['avg_preds']:>9.2f}")

System      Macro F1  Micro F1  Samples F1 Avg preds
--------------------------------------------------
Rule          0.9576    0.9544      0.6390      1.53
ML            0.9187    0.9368      0.6146      1.31
Hybrid        0.9437    0.9462      0.6368      1.55


## 6. Hybrid Delta Analysis

How many test predictions does the hybrid *actually* change relative to the ML-only
baseline — and do those changes help?

In [17]:
diff_mask = (hybrid_preds != ml_preds).any(axis=1)
n_changed = int(diff_mask.sum())
print(f"Hybrid changed prediction on {n_changed}/{len(test_texts)} test samples "
      f"({n_changed / len(test_texts):.1%})")

if n_changed > 0:
    helped = hurt = 0
    for i in np.where(diff_mask)[0]:
        ml_f1 = f1_score(test_labels[i:i + 1], ml_preds[i:i + 1], average="samples", zero_division=0)
        hy_f1 = f1_score(test_labels[i:i + 1], hybrid_preds[i:i + 1], average="samples", zero_division=0)
        if hy_f1 > ml_f1:
            helped += 1
        elif hy_f1 < ml_f1:
            hurt += 1
    print(f"  → helped {helped} samples, hurt {hurt}, neutral {n_changed - helped - hurt}")
else:
    print("  → the rule-based override did not change any ML prediction on the test set.")
    print("    This would mean the hybrid system is a no-op here — the config should be")

Hybrid changed prediction on 31/158 test samples (19.6%)
  → helped 17 samples, hurt 13, neutral 1


## 7. Per-Allergen Hybrid Modes

In [18]:
print(f"{'Allergen':<10} {'System':<8} {'Prec':>6} {'Rec':>6} {'F1':>6}")
for a in ["tree_nuts", "shellfish", "fish"]:
    j = allergen_list.index(a)
    for name, preds in [("ML", ml_preds), ("Rule", rule_preds), ("Hybrid", hybrid_preds)]:
        print(f"{a:<10} {name:<8} "
              f"{precision_score(test_labels[:, j], preds[:, j], zero_division=0):>6.3f} "
              f"{recall_score(test_labels[:, j], preds[:, j], zero_division=0):>6.3f} "
              f"{f1_score(test_labels[:, j], preds[:, j], zero_division=0):>6.3f}")
    print()

Allergen   System     Prec    Rec     F1
tree_nuts  ML        0.889  1.000  0.941
tree_nuts  Rule      0.889  1.000  0.941
tree_nuts  Hybrid    0.889  1.000  0.941

shellfish  ML        0.875  0.875  0.875
shellfish  Rule      1.000  1.000  1.000
shellfish  Hybrid    0.889  1.000  0.941

fish       ML        1.000  1.000  1.000
fish       Rule      1.000  1.000  1.000
fish       Hybrid    1.000  1.000  1.000



## 8. Tree Nuts Case Study (rare class)

tree_nuts has the fewest positives, so rule-priority (recall) vs ML (precision) tension
is most visible here. False negatives are the safety-critical failure mode for allergens.

In [19]:
j = allergen_list.index("tree_nuts")
true_tn = test_labels[:, j]
print(f"Tree nuts positives in test set: {int(true_tn.sum())}")
for name, preds in [("Rule", rule_preds), ("ML", ml_preds), ("Hybrid", hybrid_preds)]:
    print(f"  {name:8s} P={precision_score(true_tn, preds[:, j], zero_division=0):.3f}  "
          f"R={recall_score(true_tn, preds[:, j], zero_division=0):.3f}  "
          f"F1={f1_score(true_tn, preds[:, j], zero_division=0):.3f}")

fn = np.where((true_tn == 1) & (hybrid_preds[:, j] == 0))[0]
fp = np.where((true_tn == 0) & (hybrid_preds[:, j] == 1))[0]
print(f"\nHybrid tree-nuts: {len(fn)} false negatives, {len(fp)} false positives")
print("False negatives (missed tree nuts — safety critical):")
for i in list(fn[:3]):
    print(f"  - {test_texts[i][:160]}...")
print("False positives:")
for i in list(fp[:3]):
    print(f"  - {test_texts[i][:160]}...")

Tree nuts positives in test set: 8
  Rule     P=0.889  R=1.000  F1=0.941
  ML       P=0.889  R=1.000  F1=0.941
  Hybrid   P=0.889  R=1.000  F1=0.941

Hybrid tree-nuts: 0 false negatives, 1 false positives
False negatives (missed tree nuts — safety critical):
False positives:
  - wheat flour, vegetable shortening (palm oil and coconut oil), sugar, caramel chips {sugar, milk powder, hydrogenated palm kernel oil, cocoa powder, emulsifier [...


## 9. Save Production Hybrid Config

In [20]:
final_config = {
    "ml_thresholds": ml_thresholds.tolist(),
    "rule_conf_threshold": rule_conf,
    "mode": "hard_override",
    # Recall-first: the deployed system warns if EITHER the rule engine or the ML model
    # detects an allergen (union). False negatives are the safety-critical error.
    "per_allergen_modes": RECALL_FIRST_MODES,
    "max_length": max_length,
}
with open(os.path.join(dirs["models"], "hybrid_config.json"), "w") as f:
    json.dump(final_config, f, indent=2)
print("Production hybrid_config.json saved (recall-first: rule_priority for all).")

print("\nNext: run 11_model_export.ipynb for ONNX export, then 12_mobile_benchmark.ipynb.")

Production hybrid_config.json saved (recall-first: rule_priority for all).

Next: run 11_model_export.ipynb for ONNX export, then 12_mobile_benchmark.ipynb.
